In [1]:
cfg = dict(
    # Paper max length is 56, but keep yours for fair comparison
    seq_length  = 160,
    # ---------------- model (paper-aligned) ----------------
    d_model     = 512,
    latent_dim  = 64,
    enc_heads   = 8,
    dec_heads   = 8,
    enc_layers  = 7,
    dec_layers  = 7,
    ff_dim      = 1024,
    dropout     = 0.05,


    # graph feature dims (your implementation)
    node_feat_dim = 19,
    edge_feat_dim = 6,

    # special token indices
    pad_idx     = 0,
    sos_idx     = 2,
    eos_idx     = 3,

    label_smoothing = 0.02,
    corruption_p     = 0.05,

    # -------- validation / decoding --------
    beam_every  = 5,
    beam_size   = 5,
    )

In [2]:
import torch, torch.nn as nn
import model_bs as mdl
import data_utils as du

# --- paths/config you already have ---
vocab_path   = "/home/mmondol/VAE_LLM/LSTM_VAE_Paper/vocab.json"
ckpt_path    = "/home/mmondol/VAE_LLM/LSTM_VAE_Paper/checkpoints/best_model.pth"
test_csv     = "/home/mmondol/VAE_LLM/Data/Test.csv"
dye_csv      = "/home/mmondol/VAE_LLM/Data/dye.csv"

# --- load vocab ---
token_to_idx, idx_to_token = du.load_or_create_vocabulary(csv_paths=[], cache_path=vocab_path, test_smiles=None)
assert token_to_idx["<PAD>"] == cfg["pad_idx"]
assert token_to_idx["<SOS>"] == cfg["sos_idx"]
assert token_to_idx["<EOS>"] == cfg["eos_idx"]

# --- build the same architecture you trained ---
model = mdl.TVAE(
        vocab_size=len(token_to_idx),
        d_model=cfg["d_model"],
        latent_dim=cfg["latent_dim"],
        pad_idx=cfg["pad_idx"],
        sos_idx=cfg["sos_idx"],
        eos_idx=cfg["eos_idx"],
        enc_layers=cfg["enc_layers"],
        dec_layers=cfg["dec_layers"],
        enc_heads=cfg["enc_heads"],
        dec_heads=cfg["dec_heads"],
        dropout=cfg["dropout"],
        max_len=cfg["seq_length"],
        dim_feedforward=cfg["ff_dim"] if "ff_dim" in cfg else None,
        node_feat_dim=cfg.get("node_feat_dim"),
        edge_feat_dim=cfg.get("edge_feat_dim")
    )

# --- load weights robustly (handles 'module.' prefixes if any) ---
state = torch.load(ckpt_path, map_location="cpu")
try:
    model.load_state_dict(state, strict=True)
except RuntimeError:
    # remove a leading 'module.' if the checkpoint came from DataParallel
    from collections import OrderedDict
    new_state = OrderedDict()
    for k, v in state.items():
        new_state[k.replace("module.", "", 1)] = v
    model.load_state_dict(new_state, strict=True)

# --- device & optional DataParallel for speed (not required) ---
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)
print("Trainable params:", du.count_parameters(model))
print(f"Encoder parameters: {du.count_parameters(model.encoder)}")
model.eval()

/home/mmondol/anaconda3/envs/chemoinf/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[vocab] loaded cached vocabulary from /home/mmondol/VAE_LLM/LSTM_VAE_Paper/vocab.json (69 tokens)
Trainable params: 25979077
Encoder parameters: 3725312


/tmp/ipykernel_675287/2743063001.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(ckpt_path, map_location="cpu")


TVAE(
  (encoder): GraphEncoderGAT(
    (node_in): Linear(in_features=19, out_features=512, bias=True)
    (convs): ModuleList(
      (0-6): 7 x GATv2Conv(512, 64, heads=8)
    )
    (norms): ModuleList(
      (0-6): 7 x LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    )
    (out_ln): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (pool_ln): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (to_mu): Linear(in_features=512, out_features=64, bias=True)
  (to_logvar): Linear(in_features=512, out_features=64, bias=True)
  (latent_to_token): Sequential(
    (0): Linear(in_features=64, out_features=512, bias=True)
    (1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (decoder): TransformerDecoder(
    (emb): Embedding(69, 512, padding_idx=0)
    (emb_ln): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (pe): PositionalEncoding(
      (dropout): Dropout(p=0.05, inplace=False)
    )
    (dec): TransformerDecoder(
      (layers): ModuleList

In [3]:
import pandas as pd
from inference import reconstruct_smiles_table, tensor_to_smiles
import metrics as met

# Use the *unwrapped* model object for beam_search
m = model  # (if you ever wrap with DataParallel, use: model.module)

df_rec = reconstruct_smiles_table(
    smiles_list=None,
    test_csv=test_csv,
    model=m,
    token_to_idx=token_to_idx,
    idx_to_token=idx_to_token,
    seq_length=cfg["seq_length"],
    pad_idx=cfg["pad_idx"],
    sos_idx=cfg["sos_idx"],
    eos_idx=cfg["eos_idx"],
    device=device,
    mode="beam",
    beam_size=cfg["beam_size"])

# show a preview
display(df_rec.head(10))

# ------------------------------------------------------------------
# 1.  Token-level accuracy (micro-average over SMILES tokens)
# ------------------------------------------------------------------
def token_accuracy_row(gold, pred):
    g = du.tokenize_smiles(gold)
    p = du.tokenize_smiles(pred)
    L = min(len(g), len(p))
    if L == 0:                      # degenerate empty case
        return 0, 0
    correct = sum(gi == pi for gi, pi in zip(g[:L], p[:L]))
    total   = L
    return correct, total

tot_corr = tot_tok = 0
for g, p in zip(df_rec["input"], df_rec["reconstructed"]):
    c, t = token_accuracy_row(g, p)
    tot_corr += c
    tot_tok  += t

beam_token_acc = tot_corr / tot_tok if tot_tok else 0.0
print(f"Token level test accuracy (beam): {beam_token_acc:.4f}")

# ------------------------------------------------------------------
# 2.  Sequence-level (exact-match) accuracy
# ------------------------------------------------------------------
exact_match_acc = (df_rec["input"] == df_rec["reconstructed"]).mean()
print(f"Exact SMILES match accuracy (beam): {exact_match_acc:.4f}")

# ---- summary metrics (no retraining) ----
valid_ratio = (df_rec["valid"] == "yes").mean() if len(df_rec) else float("nan")
avg_lev     = df_rec["lev"].mean() if len(df_rec) else float("nan")

print(f"[beam] validity ratio: {valid_ratio:.3f}")
print(f"[beam] average Levenshtein: {avg_lev:.3f}")

/home/mmondol/anaconda3/envs/chemoinf/lib/python3.12/site-packages/torch/nn/functional.py:5193: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


,input,reconstructed,valid,lev
0,CCOC(=O)C1(CC2CC2)CC[NH+]([C@@H](C)CCC2=c3cccc...,CCOC(=O)C1(C)CC[NH+](C[C@@H]2CCCc3ccccc32)CC1,yes,18
1,O=C(N[C@@H]1CCN(CC(F)(F)F)C1=O)c1c[nH]c2cccc(F...,O=C(N[C@@H]1CCN(CC(F)(F)F)C1=O)c1c[nH]c2c(F)cc...,yes,6
2,Cc1ccc(C[NH+]2CCC[C@@H]2c2ccc(C(=O)Nc3nc(C)n(C...,Cc1nc([C@@H]2CCC[NH+]2Cc2ccc(NC(=O)c3nc(C)no3)...,yes,22
3,O=C(C[C@@H]1C(=O)N=C2[N-]c3ccccc3N21)Nc1cccc(O...,O=C1Nc2cccc(OC(F)(F)F)c2[N-]C2=C1C(=O)N(CC(=O)...,yes,45
4,CCc1ccc(-c2nc(CSc3nnnn3C3CC3)cs2)cc1,CCc1ccc(-c2nnc(SCc3csc(C4CC4)n3)n2)cc1,no,14
5,C[C@H]1CN(C(=O)N[C@@H](C)c2cccc(Br)c2)CCO1,C[C@@H]1CN(C(=O)N[C@@H](C)c2cccc(Br)c2)CCO1,yes,1
6,Cc1ccn(C)c1[C@](O)(C(=O)[O-])C(F)(F)F,Cc1ccc([C@@](O)(C(=O)[O-])C(F)(F)F)n(C)c1,no,13
7,CC(C)N(C[C@@H]1CCCCO1)C(=O)[C@H]1CCc2n[nH]c(C(...,CC(C)N(C[C@H]1CCCO1)C(=O)[C@@H]1CCCc2[nH]nc(C(...,yes,7
8,N[C@H]1C=C[C@H](c2nnc3c(Cl)cc(C(F)(F)F)cn23)C1,N[C@@H]1C=C[C@H](c2nnc3cc(C(F)(F)F)cc(Cl)c23)C1,no,12
9,CC[C@@H](C)N(Cc1c(-c2ccccc2F)noc1N1CCOCC1)C(=O...,CC[C@H](C)N(Cc1c(-c2ccco2)noc1C(=O)N1CCOCC1)c1...,yes,17


Token level test accuracy (beam): 0.7098
Exact SMILES match accuracy (beam): 0.2296
[beam] validity ratio: 0.956
[beam] average Levenshtein: 8.022


In [4]:
import pandas as pd
from inference import reconstruct_smiles_table, tensor_to_smiles
import metrics as met

# Use the *unwrapped* model object for beam_search
m = model  # (if you ever wrap with DataParallel, use: model.module)

df_rec = reconstruct_smiles_table(
    smiles_list=None,
    test_csv=dye_csv,
    model=m,
    token_to_idx=token_to_idx,
    idx_to_token=idx_to_token,
    seq_length=cfg["seq_length"],
    pad_idx=cfg["pad_idx"],
    sos_idx=cfg["sos_idx"],
    eos_idx=cfg["eos_idx"],
    device=device,
    mode="beam",
    beam_size=cfg["beam_size"])

# show a preview
display(df_rec.head(40))

# ------------------------------------------------------------------
# 1.  Token-level accuracy (micro-average over SMILES tokens)
# ------------------------------------------------------------------
def token_accuracy_row(gold, pred):
    g = du.tokenize_smiles(gold)
    p = du.tokenize_smiles(pred)
    L = min(len(g), len(p))
    if L == 0:                      # degenerate empty case
        return 0, 0
    correct = sum(gi == pi for gi, pi in zip(g[:L], p[:L]))
    total   = L
    return correct, total

tot_corr = tot_tok = 0
for g, p in zip(df_rec["input"], df_rec["reconstructed"]):
    c, t = token_accuracy_row(g, p)
    tot_corr += c
    tot_tok  += t

beam_token_acc = tot_corr / tot_tok if tot_tok else 0.0
print(f"Token level test accuracy (beam): {beam_token_acc:.4f}")

# ------------------------------------------------------------------
# 2.  Sequence-level (exact-match) accuracy
# ------------------------------------------------------------------
exact_match_acc = (df_rec["input"] == df_rec["reconstructed"]).mean()
print(f"Exact SMILES match accuracy (beam): {exact_match_acc:.4f}")

# ---- summary metrics (no retraining) ----
valid_ratio = (df_rec["valid"] == "yes").mean() if len(df_rec) else float("nan")
avg_lev     = df_rec["lev"].mean() if len(df_rec) else float("nan")

print(f"[beam] validity ratio: {valid_ratio:.3f}")
print(f"[beam] average Levenshtein: {avg_lev:.3f}")

/home/mmondol/anaconda3/envs/chemoinf/lib/python3.12/site-packages/torch/nn/functional.py:5193: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


,input,reconstructed,valid,lev
0,O=S1(=O)OC(c2ccc(O)c(Cl)c2)(c2ccc(O)c(Cl)c2)c2...,Oc1ccc([C@@H]2c3ccccc3OS(=O)(=O)c3ccc(Cl)c(Cl)...,no,34
1,O=S1(=O)OC(c2ccc(O)c(Br)c2)(c2ccc(O)c(Br)c2)c2...,Oc1cc([C@H]2Oc3ccccc3S(=O)(=O)c3ccc(Br)cc32)ccc1O,yes,34
2,Cc1cc(C2(c3cc(C)c(O)c(Br)c3)OS(=O)(=O)c3ccccc3...,Cc1cc(C)c([C@H](O)c2ccc(Br)c(Br)c2)c(Br)c1O,yes,29
3,O=S1(=O)OC(c2ccc(O)cc2)(c2ccc(O)cc2)c2ccccc21,O=c1ccc(S(=O)(=O)Oc2ccc(O)cc2)c2ccccc12,no,18
4,Cc1cc(O)ccc1C1(c2ccc(O)cc2C)OS(=O)(=O)c2ccccc21,Cc1cc(O)ccc1S(=O)(=O)Oc1ccccc1[C@H]1c2ccc(O)cc2C,no,26
5,CN(C)c1ccc(N=Nc2ccccc2)cc1,CN(C)c1ccc(Nc2ccccc2)cc1,yes,2
6,CN(C)c1ccc(N=Nc2ccc(S(=O)(=O)[O-])cc2)cc1,CN(C)c1ccc(Nc2ccc(S(=O)(=O)[O-])cc2)cc1,yes,2
7,c1ccc(-c2cc3nc4cccc(-c5ccccc5-c5ccccc5)c4nc3cc...,c1ccc2c(-c3cccc4ccccc34)c[nH]c2c1,yes,34
8,CN(C)c1ccc(N=Nc2ccccc2C(=O)O)cc1,CN(C)c1ccc(Nc2ccccc2C(=O)O)cc1,yes,2
9,O=C1OC2(c3ccc(O)cc3Oc3cc(O)ccc32)c2ccccc21,O=C1c2cc(O)ccc2Oc2ccccc21,yes,19


Token level test accuracy (beam): 0.5000
Exact SMILES match accuracy (beam): 0.0294
[beam] validity ratio: 0.853
[beam] average Levenshtein: 16.529
